<a href="https://colab.research.google.com/github/sonhh-3626/data-science-economics/blob/main/clean-and-normalization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Khởi tạo bộ dữ liệu nhân sự

In [7]:
import pandas as pd
import numpy as np

# Khởi tạo tập dữ liệu nhân sự có chứa lỗi
data_nhansu = {
    'MaNV': ['NV01', 'NV02', 'NV03', 'NV04', 'NV02', 'NV05', 'NV06'],
    'HoTen': ['Nguyễn Văn An', 'Trần Thị Bình', 'Lê Văn Cường', 'Phạm Minh Đông', 'Trần Thị Bình', 'Hoàng Anh', 'Vũ Nam'],
    'Tuoi': [25, np.nan, 30, 28, np.nan, 35, 22],
    'Luong_USD': [1500, 1800, np.nan, 2200, 1800, 3500, 1200],
    'PhongBan': ['IT', 'Marketing', None, 'IT', 'Marketing', 'Sales', 'IT']
}
df = pd.DataFrame(data_nhansu)
display(df)

,MaNV,HoTen,Tuoi,Luong_USD,PhongBan
0,NV01,Nguyễn Văn An,25.0,1500.0,IT
1,NV02,Trần Thị Bình,NaN,1800.0,Marketing
2,NV03,Lê Văn Cường,30.0,NaN,None
3,NV04,Phạm Minh Đông,28.0,2200.0,IT
4,NV02,Trần Thị Bình,NaN,1800.0,Marketing
5,NV05,Hoàng Anh,35.0,3500.0,Sales
6,NV06,Vũ Nam,22.0,1200.0,IT


### 1. Phát hiện & Đánh giá dữ liệu

In [8]:
# Kiểm tra các ô dữ liệu bị thiếu
missing_values = df.isnull().sum()
print("Số lượng giá trị khuyết ở từng cột:")
print(missing_values)

# Kiểm tra dữ liệu bị trùng lặp
duplicate_rows = df[df.duplicated(keep=False)]
print("\nCác dòng dữ liệu bị trùng lặp:")
display(duplicate_rows)

Số lượng giá trị khuyết ở từng cột:
MaNV         0
HoTen        0
Tuoi         2
Luong_USD    1
PhongBan     1
dtype: int64

Các dòng dữ liệu bị trùng lặp:


,MaNV,HoTen,Tuoi,Luong_USD,PhongBan
1,NV02,Trần Thị Bình,NaN,1800.0,Marketing
4,NV02,Trần Thị Bình,NaN,1800.0,Marketing


### 2. Xử lý dữ liệu thiếu & trùng lặp

In [9]:
# Loại bỏ các dòng trùng lặp (giữ lại dòng đầu tiên - keep='first')
# Lý do chọn keep='first': Giúp giữ bản ghi gốc được tạo ra trước, tránh thay đổi thứ tự log tự nhiên của hệ thống
df_cleaned = df.drop_duplicates(keep='first').copy()

# Điền khuyết cho cột 'Tuoi' bằng Median
median_tuoi = df_cleaned['Tuoi'].median()
df_cleaned['Tuoi'] = df_cleaned['Tuoi'].fillna(median_tuoi)

# Điền khuyết cho cột 'Luong_USD' bằng Mean
mean_luong = df_cleaned['Luong_USD'].mean()
df_cleaned['Luong_USD'] = df_cleaned['Luong_USD'].fillna(mean_luong)

# Điền khuyết cho cột 'PhongBan' bằng ffill (forward fill)
df_cleaned['PhongBan'] = df_cleaned['PhongBan'].ffill()

display(df_cleaned)

,MaNV,HoTen,Tuoi,Luong_USD,PhongBan
0,NV01,Nguyễn Văn An,25.0,1500.0,IT
1,NV02,Trần Thị Bình,28.0,1800.0,Marketing
2,NV03,Lê Văn Cường,30.0,2040.0,Marketing
3,NV04,Phạm Minh Đông,28.0,2200.0,IT
5,NV05,Hoàng Anh,35.0,3500.0,Sales
6,NV06,Vũ Nam,22.0,1200.0,IT


### 3. Chuẩn hóa dữ liệu

In [10]:
# Áp dụng Min-Max Scaling cho cột Luong_USD về khoảng [0, 1]
min_l = df_cleaned['Luong_USD'].min()
max_l = df_cleaned['Luong_USD'].max()
df_cleaned['Luong_MinMax'] = (df_cleaned['Luong_USD'] - min_l) / (max_l - min_l)

# So sánh Min-Max Scaling và Z-score Standardization đối với cột Tuoi
# Min-Max Scaling cột Tuoi
min_t = df_cleaned['Tuoi'].min()
max_t = df_cleaned['Tuoi'].max()
df_cleaned['Tuoi_MinMax'] = (df_cleaned['Tuoi'] - min_t) / (max_t - min_t)

# Z-score Standardization cột Tuoi
mean_t = df_cleaned['Tuoi'].mean()
std_t = df_cleaned['Tuoi'].std()
df_cleaned['Tuoi_Zscore'] = (df_cleaned['Tuoi'] - mean_t) / std_t

display(df_cleaned[['HoTen', 'Tuoi', 'Tuoi_MinMax', 'Tuoi_Zscore', 'Luong_USD', 'Luong_MinMax']])

,HoTen,Tuoi,Tuoi_MinMax,Tuoi_Zscore,Luong_USD,Luong_MinMax
0,Nguyễn Văn An,25.0,0.230769,-0.677631,1500.0,0.130435
1,Trần Thị Bình,28.0,0.461538,0.000000,1800.0,0.260870
2,Lê Văn Cường,30.0,0.615385,0.451754,2040.0,0.365217
3,Phạm Minh Đông,28.0,0.461538,0.000000,2200.0,0.434783
5,Hoàng Anh,35.0,1.000000,1.581139,3500.0,1.000000
6,Vũ Nam,22.0,0.000000,-1.355262,1200.0,0.000000
